In [2]:
import pandas as pd
import numpy as np

In [2]:
# LOad data
df = pd.read_csv("../data/f1_data.csv")
print(df.shape)
print(df.head())

(6915, 10)
   season  round           race_name  circuit    driver      team  grid  \
0    2010      1  Bahrain Grand Prix  bahrain    alonso   Ferrari     3   
1    2010      1  Bahrain Grand Prix  bahrain     massa   Ferrari     2   
2    2010      1  Bahrain Grand Prix  bahrain  hamilton   McLaren     4   
3    2010      1  Bahrain Grand Prix  bahrain    vettel  Red Bull     1   
4    2010      1  Bahrain Grand Prix  bahrain   rosberg  Mercedes     5   

  position  points    status  
0        1    25.0  Finished  
1        2    18.0  Finished  
2        3    15.0  Finished  
3        4    12.0  Finished  
4        5    10.0  Finished  


In [3]:
# Fix data types and sort
df["position"] = pd.to_numeric(df["position"], errors="coerce")
df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
df["points"] = pd.to_numeric(df["points"], errors="coerce")

# sort by season and round 
df = df.sort_values(["season", "round"]).reset_index(drop=True)

print(df.dtypes)

season         int64
round          int64
race_name     object
circuit       object
driver        object
team          object
grid           int64
position     float64
points       float64
status        object
dtype: object


In [4]:
# 1 if this driver won this race, 0 if not
df["won"] = (df["position"] == 1).astype(int)

print(f"Total wins: {df['won'].sum()}")
print(f"Win rate: {df['won'].mean():.3f}")

Total wins: 329
Win rate: 0.048


In [ ]:
# FEATURE 1 — driver average finish position over last 3 races
df["driver_avg_finish_last3"] = (
    df.groupby("driver")["position"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

print(df[["season", "round", "driver", "position", "driver_avg_finish_last3"]].head(20))

    season  round              driver  position  driver_avg_finish_last3
0     2010      1              alonso       1.0                      NaN
1     2010      1               massa       2.0                      NaN
2     2010      1            hamilton       3.0                      NaN
3     2010      1              vettel       4.0                      NaN
4     2010      1             rosberg       5.0                      NaN
5     2010      1  michael_schumacher       6.0                      NaN
6     2010      1              button       7.0                      NaN
7     2010      1              webber       8.0                      NaN
8     2010      1              liuzzi       9.0                      NaN
9     2010      1         barrichello      10.0                      NaN
10    2010      1              kubica      11.0                      NaN
11    2010      1               sutil      12.0                      NaN
12    2010      1         alguersuari      13.0    

In [6]:
# check verstappen's rolling average across multiple races to verify it's working
verstappen = df[df["driver"] == "max_verstappen"][["season", "round", "position", "driver_avg_finish_last3"]].head(20)
print(verstappen)

      season  round  position  driver_avg_finish_last3
2229    2015      1       NaN                      NaN
2241    2015      2       7.0                      NaN
2271    2015      3      17.0                 7.000000
2292    2015      4       NaN                12.000000
2305    2015      5      11.0                12.000000
2332    2015      6       NaN                14.000000
2349    2015      7      15.0                11.000000
2362    2015      8       8.0                13.000000
2390    2015      9       NaN                11.500000
2398    2015     10       4.0                11.500000
2422    2015     11       8.0                 6.000000
2446    2015     12      12.0                 6.000000
2462    2015     13       8.0                 8.000000
2483    2015     14       9.0                 9.333333
2503    2015     15      10.0                 9.666667
2518    2015     16       4.0                 9.000000
2543    2015     17       9.0                 7.666667
2563    20

In [ ]:
# FEATURE 2 — Driver win rate at this specific circuit

df["driver_circuit_win_rate"] = (
    df.groupby(["driver", "circuit"])["won"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print(df[["season", "round", "circuit", "driver", "won", "driver_circuit_win_rate"]].head(20))

    season  round  circuit              driver  won  driver_circuit_win_rate
0     2010      1  bahrain              alonso    1                      NaN
1     2010      1  bahrain               massa    0                      NaN
2     2010      1  bahrain            hamilton    0                      NaN
3     2010      1  bahrain              vettel    0                      NaN
4     2010      1  bahrain             rosberg    0                      NaN
5     2010      1  bahrain  michael_schumacher    0                      NaN
6     2010      1  bahrain              button    0                      NaN
7     2010      1  bahrain              webber    0                      NaN
8     2010      1  bahrain              liuzzi    0                      NaN
9     2010      1  bahrain         barrichello    0                      NaN
10    2010      1  bahrain              kubica    0                      NaN
11    2010      1  bahrain               sutil    0                      NaN

In [8]:
# check verstappen at monaco where he has history
monaco_ver = df[(df["driver"] == "max_verstappen") & (df["circuit"] == "monaco")][
    ["season", "round", "circuit", "driver", "won", "driver_circuit_win_rate"]
]
print(monaco_ver)

      season  round circuit          driver  won  driver_circuit_win_rate
2332    2015      6  monaco  max_verstappen    0                      NaN
2722    2016      6  monaco  max_verstappen    0                 0.000000
3161    2017      6  monaco  max_verstappen    0                 0.000000
3565    2018      6  monaco  max_verstappen    0                 0.000000
3980    2019      6  monaco  max_verstappen    0                 0.000000
4718    2021      5  monaco  max_verstappen    1                 0.000000
5199    2022      7  monaco  max_verstappen    0                 0.166667
5617    2023      6  monaco  max_verstappen    1                 0.142857
6101    2024      8  monaco  max_verstappen    0                 0.250000
6579    2025      8  monaco  max_verstappen    0                 0.222222


In [ ]:
# FEATURE 3 — Constructor points last 3 races
team_points = df.groupby(["season", "round", "team"])["points"].sum().reset_index()
team_points["team_avg_points_last3"] = (
    team_points.groupby(["season", "team"])["points"]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# merge back into main dataframe
df = df.drop(columns=["team_avg_points_last3"])
df = df.merge(team_points[["season", "round", "team", "team_avg_points_last3"]], on=["season", "round", "team"])

print(df[["season", "round", "driver", "team", "points", "team_avg_points_last3"]].head(20))

    season  round              driver         team  points  \
0     2010      1              alonso      Ferrari    25.0   
1     2010      1               massa      Ferrari    18.0   
2     2010      1            hamilton      McLaren    15.0   
3     2010      1              vettel     Red Bull    12.0   
4     2010      1             rosberg     Mercedes    10.0   
5     2010      1  michael_schumacher     Mercedes     8.0   
6     2010      1              button      McLaren     6.0   
7     2010      1              webber     Red Bull     4.0   
8     2010      1              liuzzi  Force India     2.0   
9     2010      1         barrichello     Williams     1.0   
10    2010      1              kubica      Renault     0.0   
11    2010      1               sutil  Force India     0.0   
12    2010      1         alguersuari   Toro Rosso     0.0   
13    2010      1          hulkenberg     Williams     0.0   
14    2010      1          kovalainen        Lotus     0.0   
15    20

In [12]:
# check red bull's team momentum across 2023 season
red_bull_2023 = df[(df["team"] == "Red Bull") & (df["season"] == 2023)][
    ["season", "round", "driver", "team", "points", "team_avg_points_last3"]
].head(20)
print(red_bull_2023)

      season  round          driver      team  points  team_avg_points_last3
5517    2023      1  max_verstappen  Red Bull    25.0                    NaN
5518    2023      1           perez  Red Bull    18.0                    NaN
5537    2023      2           perez  Red Bull    25.0              43.000000
5538    2023      2  max_verstappen  Red Bull    19.0              43.000000
5557    2023      3  max_verstappen  Red Bull    25.0              43.500000
5561    2023      3           perez  Red Bull    11.0              43.500000
5577    2023      4           perez  Red Bull    25.0              41.000000
5578    2023      4  max_verstappen  Red Bull    18.0              41.000000
5597    2023      5  max_verstappen  Red Bull    26.0              41.000000
5598    2023      5           perez  Red Bull    18.0              41.000000
5617    2023      6  max_verstappen  Red Bull    25.0              41.000000
5632    2023      6           perez  Red Bull     0.0              41.000000

In [ ]:
# FEATURE 4 — Driver DNF rate
finished_statuses = ["Finished", "Lapped", "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps"]
df["dnf"] = (~df["status"].isin(finished_statuses)).astype(int)

df["driver_dnf_rate"] = (
    df.groupby("driver")["dnf"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

print(df[["season", "round", "driver", "status", "dnf", "driver_dnf_rate"]].head(20))

    season  round              driver       status  dnf  driver_dnf_rate
0     2010      1              alonso     Finished    0              NaN
1     2010      1               massa     Finished    0              NaN
2     2010      1            hamilton     Finished    0              NaN
3     2010      1              vettel     Finished    0              NaN
4     2010      1             rosberg     Finished    0              NaN
5     2010      1  michael_schumacher     Finished    0              NaN
6     2010      1              button     Finished    0              NaN
7     2010      1              webber     Finished    0              NaN
8     2010      1              liuzzi     Finished    0              NaN
9     2010      1         barrichello     Finished    0              NaN
10    2010      1              kubica     Finished    0              NaN
11    2010      1               sutil     Finished    0              NaN
12    2010      1         alguersuari     Finished 

In [ ]:
# FEATURE 5 — regulation era flag

def get_regulation_era(season):
    if season <= 2013:
        return 0  # V8 era
    elif season <= 2021:
        return 1  # V6 hybrid era
    elif season <= 2025:
        return 2  # ground effect era
    else:
        return 3  # 2026 new hybrid era

df["regulation_era"] = df["season"].apply(get_regulation_era)

print(df[["season", "regulation_era"]].drop_duplicates().sort_values("season"))

      season  regulation_era
0       2010               0
456     2011               0
912     2012               0
1392    2013               0
1810    2014               1
2217    2015               1
2595    2016               1
3057    2017               1
3457    2018               1
3877    2019               1
4297    2020               1
4637    2021               1
5077    2022               2
5517    2023               2
5957    2024               2
6436    2025               2


In [16]:
# FEATURE 6 — driver championship standing at time of race
df["driver_championship_pos"] = (
    df.groupby(["season", "round"])["points"]
    .rank(ascending=False, method="min")
)

df["cumulative_points"] = (
    df.groupby(["season", "driver"])["points"]
    .transform(lambda x: x.shift(1).expanding().sum())
)

df["driver_championship_pos"] = (
    df.groupby(["season", "round"])["cumulative_points"]
    .rank(ascending=False, method="min")
)

print(df[["season", "round", "driver", "cumulative_points", "driver_championship_pos"]].head(20))

    season  round              driver  cumulative_points  \
0     2010      1              alonso                NaN   
1     2010      1               massa                NaN   
2     2010      1            hamilton                NaN   
3     2010      1              vettel                NaN   
4     2010      1             rosberg                NaN   
5     2010      1  michael_schumacher                NaN   
6     2010      1              button                NaN   
7     2010      1              webber                NaN   
8     2010      1              liuzzi                NaN   
9     2010      1         barrichello                NaN   
10    2010      1              kubica                NaN   
11    2010      1               sutil                NaN   
12    2010      1         alguersuari                NaN   
13    2010      1          hulkenberg                NaN   
14    2010      1          kovalainen                NaN   
15    2010      1               buemi   

In [17]:
# check championship standings after a few races in 2023
champ_2023 = df[(df["season"] == 2023) & (df["round"] == 5)][
    ["season", "round", "driver", "cumulative_points", "driver_championship_pos"]
].sort_values("driver_championship_pos")
print(champ_2023)

      season  round           driver  cumulative_points  \
5597    2023      5   max_verstappen               87.0   
5598    2023      5            perez               79.0   
5599    2023      5           alonso               57.0   
5602    2023      5         hamilton               46.0   
5601    2023      5            sainz               30.0   
5608    2023      5           stroll               26.0   
5600    2023      5          russell               23.0   
5603    2023      5          leclerc               21.0   
5613    2023      5           norris               10.0   
5611    2023      5       hulkenberg                6.0   
5609    2023      5           bottas                4.0   
5604    2023      5            gasly                4.0   
5615    2023      5          piastri                4.0   
5605    2023      5             ocon                4.0   
5612    2023      5             zhou                2.0   
5607    2023      5          tsunoda                2.0 

In [18]:
# save progress
df.to_csv("../data/f1_features.csv", index=False)
print("Saved!")
print(f"\nFeatures built so far:")
print(df.columns.tolist())

Saved!

Features built so far:
['season', 'round', 'race_name', 'circuit', 'driver', 'team', 'grid', 'position', 'points', 'status', 'won', 'driver_avg_finish_last3', 'driver_circuit_win_rate', 'team_avg_points_last3', 'dnf', 'driver_dnf_rate', 'regulation_era', 'driver_championship_pos', 'cumulative_points']


In [20]:
# FEATURE 7 — home race flag
# driver nationality to home circuit mapping
home_circuits = {
    "max_verstappen": "zandvoort",
    "hamilton": "silverstone",
    "leclerc": "monaco",
    "sainz": "catalunya",
    "alonso": "catalunya",
    "norris": "silverstone",
    "russell": "silverstone",
    "vettel": "hockenheimring",
    "michael_schumacher": "hockenheimring",
    "rosberg": "hockenheimring",
    "hulkenberg": "hockenheimring",
    "bottas": "finland",  # no Finnish GP so will never trigger
    "raikkonen": "finland",
    "massa": "interlagos",
    "barrichello": "interlagos",
    "fittipaldi": "interlagos",
    "perez": "rodriguez",
    "gutierrez": "rodriguez",
    "button": "silverstone",
    "coulthard": "silverstone",
    "webber": "albert_park",
    "ricciardo": "albert_park",
    "piastri": "albert_park",
    "gasly": "paul_ricard",
    "ocon": "paul_ricard",
    "grosjean": "paul_ricard",
    "tsunoda": "suzuka",
    "kobayashi": "suzuka",
    "zhou": "shanghai",
    "stroll": "villeneuve",
    "latifi": "villeneuve",
}

df["home_race"] = df.apply(
    lambda row: 1 if home_circuits.get(row["driver"]) == row["circuit"] else 0,
    axis=1
)

print(f"Home races in dataset: {df['home_race'].sum()}")
print(df[df["home_race"] == 1][["season", "driver", "circuit"]].head(20))

Home races in dataset: 149
     season              driver         circuit
32     2010              webber     albert_park
97     2010              alonso       catalunya
217    2010            hamilton     silverstone
219    2010              button     silverstone
242    2010              vettel  hockenheimring
247    2010             rosberg  hockenheimring
248    2010  michael_schumacher  hockenheimring
252    2010          hulkenberg  hockenheimring
366    2010           kobayashi          suzuka
421    2010         barrichello      interlagos
422    2010               massa      interlagos
460    2011              webber     albert_park
556    2011              alonso       catalunya
651    2011            hamilton     silverstone
667    2011              button     silverstone
804    2011           kobayashi          suzuka
892    2011               massa      interlagos
901    2011         barrichello      interlagos
915    2012              webber     albert_park
920    2012  

In [21]:
# FEATURE 8 — gap to championship leader in points
leader_points = df.groupby(["season", "round"])["cumulative_points"].max().reset_index()
leader_points.columns = ["season", "round", "leader_points"]

df = df.merge(leader_points, on=["season", "round"])
df["gap_to_leader"] = df["leader_points"] - df["cumulative_points"]

print(df[["season", "round", "driver", "cumulative_points", "leader_points", "gap_to_leader"]].head(20))

    season  round              driver  cumulative_points  leader_points  \
0     2010      1              alonso                NaN            NaN   
1     2010      1               massa                NaN            NaN   
2     2010      1            hamilton                NaN            NaN   
3     2010      1              vettel                NaN            NaN   
4     2010      1             rosberg                NaN            NaN   
5     2010      1  michael_schumacher                NaN            NaN   
6     2010      1              button                NaN            NaN   
7     2010      1              webber                NaN            NaN   
8     2010      1              liuzzi                NaN            NaN   
9     2010      1         barrichello                NaN            NaN   
10    2010      1              kubica                NaN            NaN   
11    2010      1               sutil                NaN            NaN   
12    2010      1        

In [22]:
# check round 10 of 2023
gap_2023 = df[(df["season"] == 2023) & (df["round"] == 10)][
    ["season", "round", "driver", "cumulative_points", "leader_points", "gap_to_leader"]
].sort_values("gap_to_leader")
print(gap_2023)

      season  round           driver  cumulative_points  leader_points  \
5697    2023     10   max_verstappen              215.0          215.0   
5702    2023     10            perez              133.0          215.0   
5703    2023     10           alonso              124.0          215.0   
5699    2023     10         hamilton              104.0          215.0   
5706    2023     10            sainz               72.0          215.0   
5701    2023     10          russell               66.0          215.0   
5705    2023     10          leclerc               65.0          215.0   
5710    2023     10           stroll               38.0          215.0   
5716    2023     10             ocon               29.0          215.0   
5698    2023     10           norris               24.0          215.0   
5714    2023     10            gasly               16.0          215.0   
5704    2023     10            albon                7.0          215.0   
5709    2023     10       hulkenberg  

In [23]:
#Save final feature and drop unnessesary columns
df_model = df.drop(columns=["race_name", "status", "dnf", "cumulative_points", "leader_points"])

df_model.to_csv("../data/f1_features.csv", index=False)

print(f"Saved! Shape: {df_model.shape}")
print(f"\nFinal features:")
print(df_model.columns.tolist())

Saved! Shape: (6915, 17)

Final features:
['season', 'round', 'circuit', 'driver', 'team', 'grid', 'position', 'points', 'won', 'driver_avg_finish_last3', 'driver_circuit_win_rate', 'team_avg_points_last3', 'driver_dnf_rate', 'regulation_era', 'driver_championship_pos', 'home_race', 'gap_to_leader']
